# 10k Dataset analysis: 

In this notebook, the ten thousand patients generated with the supercluster are analysed in order to identify if the patients are physiologically realistic. Equations and conditions are verified. If enough patients pass the tests, then they will be used to train the first series of tree-searching algorithms and Neural networks. 

Physiological laws: 

- maximum of 15% difference between input and real output variables (i.e: CO and CO_real ) (modifiable depending on variable )

- LASV and RASV need to have same order of magnitude 

- LVSV and RVSV difference < 1ml
stroke volumes of ventricles need to be same to have a stable loop

- LASV and RASV difference < 1ml
stroke volumes of atria need to be same to have a stable loop

- ASV =< VSV/3
stroke volume of atria has to be less than a third of ventricle
(has to be true for each, left and right)

- LAP > PAP
due to pressure drop from pulmonary artery to atrium

- SAP > CVP
due to pressure drop from systemic artery to systemic veins

- CVP = RAP
DEFINITION in our model because no vena cava model

- abs(SAP-DAP)> 30mmHg

- abs(sPAP -DPAP)>15mmHg


### Import csv 

In [1]:
import pandas as pd 
import numpy as np 

In [4]:
from pathlib import Path

ROOT = Path.cwd().parents[0]  

csv_path = (
    ROOT
    / "01_Data"
    / "simulation_results"
    / "pediatric_dcm_patient_01"
    / "stage_1_10k_supercomputer_v2"
    / "pediatric_dcm_patient_01_stage_1_10k_supercomputer_v2_FULL_DATASET.csv"
)

df = pd.read_csv(csv_path)

tunables = [
    "expansion_factor",
    "k_Vtot",
    "k_Vsys",
    "k_Vusv_sys",
    "k_Vusv_sys_ven",
    "k_Vusv_pulm_ven",
    "k_Ctot",
    "k_Csys",
    "k_Rsysven",
    "k_Rpulmart",
    "k_ESP_LV",
    "k_ESP_RV",
]

tunables = [col for col in tunables if col in df.columns]

print("Using tunables:", tunables)

failed = df[df["simulation_status"] == "failed"]
success = df[df["simulation_status"] == "success"]

print("Loaded:", csv_path)
print("Total:", len(df))
print("Success:", len(success))
print("Failed:", len(failed))
print("Failure rate:", len(failed) / len(df) * 100, "%")

print("\nFailed cases:")
print(failed[["simulation_id"] + tunables])

print("\nMean tunables: success vs failed")
print(df.groupby("simulation_status")[tunables].mean())

print("\nMin/max tunables in failed cases")
print(failed[tunables].describe().T)

Using tunables: ['k_Vtot', 'k_Vsys', 'k_Vusv_sys', 'k_Vusv_sys_ven', 'k_Vusv_pulm_ven', 'k_Ctot', 'k_Csys', 'k_Rsysven', 'k_Rpulmart', 'k_ESP_LV', 'k_ESP_RV']
Loaded: c:\Users\Maxim\Documents\02_ESILV\A4\Stage\03_Models\DIA\Main_project\01_Data\simulation_results\pediatric_dcm_patient_01\stage_1_10k_supercomputer_v2\pediatric_dcm_patient_01_stage_1_10k_supercomputer_v2_FULL_DATASET.csv
Total: 10000
Success: 10000
Failed: 0
Failure rate: 0.0 %

Failed cases:
Empty DataFrame
Columns: [simulation_id, k_Vtot, k_Vsys, k_Vusv_sys, k_Vusv_sys_ven, k_Vusv_pulm_ven, k_Ctot, k_Csys, k_Rsysven, k_Rpulmart, k_ESP_LV, k_ESP_RV]
Index: []

Mean tunables: success vs failed
                      k_Vtot  k_Vsys  k_Vusv_sys  k_Vusv_sys_ven  \
simulation_status                                                  
success            72.499993    0.84        0.84           0.945   

                   k_Vusv_pulm_ven  k_Ctot  k_Csys  k_Rsysven  k_Rpulmart  \
simulation_status                                  

### Verifying Physiological bounds: 

In [6]:
# ============================================================
# PHYSIOLOGICAL VALIDITY CHECK
# ============================================================

TARGETS = {
    "LAP_real": 19,
    "RAP_real": 10,
    "SAP_real": 99,
    "DAP_real": 60,
    "sPAP_real": 40,
    "dPAP_real": 22,
    "EDV_LV_real": 99,
    "ESV_LV_real": 76,
    "EDV_RV_real": 63,
    "ESV_RV_real": 40,
    "CO_real": 2576 / 1000,  # L/min
}

REL_TOL = 0.15

# ------------------------------------------------------------
# 1. Relative error checks
# ------------------------------------------------------------

rule_cols = []

for col, target in TARGETS.items():

    if col not in df.columns:
        print(f"WARNING: {col} missing")
        continue

    err_col = f"{col}_relative_error"
    rule_col = f"rule_{col}_within_15pct"

    df[err_col] = np.abs(df[col] - target) / np.maximum(np.abs(target), 1e-12)

    df[rule_col] = df[err_col] <= REL_TOL

    rule_cols.append(rule_col)

# ------------------------------------------------------------
# 2. Stroke volumes
# ------------------------------------------------------------

df["LVSV_real"] = df["EDV_LV_real"] - df["ESV_LV_real"]
df["RVSV_real"] = df["EDV_RV_real"] - df["ESV_RV_real"]

# Ventricular SV consistency
df["rule_LVSV_RVSV_diff_lt_1ml"] = (
    np.abs(df["LVSV_real"] - df["RVSV_real"]) < 1
)

rule_cols.append("rule_LVSV_RVSV_diff_lt_1ml")

# ------------------------------------------------------------
# 3. Atrial stroke volumes (if available)
# ------------------------------------------------------------

possible_lasv = ["LASV_real", "LASV", "LA_SV_real"]
possible_rasv = ["RASV_real", "RASV", "RA_SV_real"]

lasv_col = next((c for c in possible_lasv if c in df.columns), None)
rasv_col = next((c for c in possible_rasv if c in df.columns), None)

if lasv_col and rasv_col:

    print(f"Using atrial SV columns: {lasv_col}, {rasv_col}")

    # Same order of magnitude
    ratio = (
        np.abs(df[lasv_col]) /
        np.maximum(np.abs(df[rasv_col]), 1e-12)
    )

    df["rule_LASV_RASV_same_order"] = ratio.between(0.1, 10)

    # Difference < 1 mL
    df["rule_LASV_RASV_diff_lt_1ml"] = (
        np.abs(df[lasv_col] - df[rasv_col]) < 1
    )

    # ASV <= VSV / 3
    df["rule_LASV_le_LVSV_over_3"] = (
        df[lasv_col] <= (df["LVSV_real"] / 3)
    )

    df["rule_RASV_le_RVSV_over_3"] = (
        df[rasv_col] <= (df["RVSV_real"] / 3)
    )

    rule_cols += [
        "rule_LASV_RASV_same_order",
        "rule_LASV_RASV_diff_lt_1ml",
        "rule_LASV_le_LVSV_over_3",
        "rule_RASV_le_RVSV_over_3",
    ]

else:
    print("No atrial stroke volume columns found -> skipping ASV rules")

# ------------------------------------------------------------
# 4. Pressure relationships
# ------------------------------------------------------------

# Mean PAP
df["mPAP_real"] = (
    df["sPAP_real"] + df["dPAP_real"]
) / 2

# Physiological pulmonary pressure drop
df["rule_mPAP_gt_LAP"] = (
    df["mPAP_real"] > df["LAP_real"]
)

rule_cols.append("rule_mPAP_gt_LAP")

# Systemic pressure drop
df["rule_SAP_gt_CVP"] = (
    df["SAP_real"] > 10
)

rule_cols.append("rule_SAP_gt_CVP")

# RAP ~= CVP
df["rule_RAP_close_to_CVP"] = (
    np.abs(df["RAP_real"] - 10) <= 1
)

rule_cols.append("rule_RAP_close_to_CVP")

# Pulse pressures
df["rule_systemic_pulse_pressure_gt_30"] = (
    np.abs(df["SAP_real"] - df["DAP_real"]) > 30
)

rule_cols.append("rule_systemic_pulse_pressure_gt_30")

df["rule_pulmonary_pulse_pressure_gt_15"] = (
    np.abs(df["sPAP_real"] - df["dPAP_real"]) > 15
)

rule_cols.append("rule_pulmonary_pulse_pressure_gt_15")

# ------------------------------------------------------------
# 5. Simulink success required
# ------------------------------------------------------------

df["rule_simulation_success"] = (
    df["simulation_status"] == "success"
)

rule_cols.append("rule_simulation_success")

# ------------------------------------------------------------
# FINAL BOOLEAN
# ------------------------------------------------------------

df["physiological_interpretation"] = df[rule_cols].all(axis=1)

# ------------------------------------------------------------
# FAILED RULES
# ------------------------------------------------------------

def failed_rules(row):
    return [
        col for col in rule_cols
        if not row[col]
    ]

df["failed_rules"] = df.apply(failed_rules, axis=1)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PHYSIOLOGICAL VALIDITY SUMMARY")
print("=" * 60)

print("Physiological:",
      df["physiological_interpretation"].sum())

print("Non-physiological:",
      (~df["physiological_interpretation"]).sum())

print("Physiological rate:",
      round(df["physiological_interpretation"].mean() * 100, 2),
      "%")

print("\nFailed rule counts:")

for col in rule_cols:
    print(f"{col}: {(~df[col]).sum()}")

# ------------------------------------------------------------
# SHOW NON-PHYSIOLOGICAL CASES
# ------------------------------------------------------------

non_phys = df[~df["physiological_interpretation"]]

print("\nFirst non-physiological simulations:")
display(
    non_phys[
        ["simulation_id", "failed_rules"]
    ].head(20)
)

No atrial stroke volume columns found -> skipping ASV rules

PHYSIOLOGICAL VALIDITY SUMMARY
Physiological: 768
Non-physiological: 9232
Physiological rate: 7.68 %

Failed rule counts:
rule_LAP_real_within_15pct: 4665
rule_RAP_real_within_15pct: 309
rule_SAP_real_within_15pct: 3948
rule_DAP_real_within_15pct: 72
rule_sPAP_real_within_15pct: 1965
rule_dPAP_real_within_15pct: 1979
rule_EDV_LV_real_within_15pct: 0
rule_ESV_LV_real_within_15pct: 2
rule_EDV_RV_real_within_15pct: 0
rule_ESV_RV_real_within_15pct: 0
rule_CO_real_within_15pct: 7
rule_LVSV_RVSV_diff_lt_1ml: 0
rule_mPAP_gt_LAP: 0
rule_SAP_gt_CVP: 0
rule_RAP_close_to_CVP: 1526
rule_systemic_pulse_pressure_gt_30: 7211
rule_pulmonary_pulse_pressure_gt_15: 2868
rule_simulation_success: 0

First non-physiological simulations:


,simulation_id,failed_rules
0,1,"[rule_SAP_real_within_15pct, rule_systemic_pul..."
1,2,[rule_systemic_pulse_pressure_gt_30]
2,3,"[rule_RAP_close_to_CVP, rule_systemic_pulse_pr..."
3,4,"[rule_SAP_real_within_15pct, rule_systemic_pul..."
4,5,"[rule_SAP_real_within_15pct, rule_systemic_pul..."
5,6,"[rule_SAP_real_within_15pct, rule_RAP_close_to..."
6,7,"[rule_SAP_real_within_15pct, rule_RAP_close_to..."
7,8,"[rule_SAP_real_within_15pct, rule_systemic_pul..."
8,9,"[rule_LAP_real_within_15pct, rule_RAP_close_to..."
9,10,"[rule_RAP_real_within_15pct, rule_SAP_real_wit..."


### Tightening the tunable parameter bounds based on the physiological realism: 

In [7]:
phys = df[df["physiological_interpretation"]].copy()
non_phys = df[~df["physiological_interpretation"]].copy()

In [8]:
new_bounds = []

for col in tunables:
    new_bounds.append({
        "variable": col,
        "old_lower": df[col].min(),
        "old_upper": df[col].max(),
        "new_lower_q01": phys[col].quantile(0.01),
        "new_upper_q99": phys[col].quantile(0.99),
        "strict_lower_q05": phys[col].quantile(0.05),
        "strict_upper_q95": phys[col].quantile(0.95),
        "phys_median": phys[col].median(),
    })

new_bounds_df = pd.DataFrame(new_bounds)
display(new_bounds_df)

,variable,old_lower,old_upper,new_lower_q01,new_upper_q99,strict_lower_q05,strict_upper_q95,phys_median
0,k_Vtot,60.001935,84.998702,60.181333,84.773704,61.395063,83.615921,72.053019
1,k_Vsys,0.750000,0.929992,0.751734,0.928422,0.757768,0.921362,0.844127
2,k_Vusv_sys,0.750012,0.929992,0.751265,0.928324,0.758933,0.920402,0.838412
3,k_Vusv_sys_ven,0.900001,0.989992,0.900853,0.989226,0.904895,0.983964,0.944223
4,k_Vusv_pulm_ven,0.800010,0.969999,0.801997,0.967670,0.806357,0.959941,0.882030
5,k_Ctot,1.720057,2.579927,1.727778,2.570081,1.758182,2.541812,2.164746
6,k_Csys,0.750013,0.929998,0.763479,0.928993,0.793538,0.925027,0.878090
7,k_Rsysven,0.048002,0.071998,0.048214,0.071824,0.049509,0.071133,0.062319
8,k_Rpulmart,0.500009,0.799993,0.504799,0.797705,0.520434,0.787949,0.677328
9,k_ESP_LV,0.750029,1.049988,0.894233,1.040333,0.904693,1.021984,0.956243


In [12]:
important_outputs = [
    "LAP_real",
    "RAP_real",
    "SAP_real",
    "DAP_real",
    "sPAP_real",
    "dPAP_real",
    "CO_real",
]

corr = df[tunables + important_outputs].corr()

display(
    corr.loc[tunables, important_outputs]
    .abs()
    
)

,LAP_real,RAP_real,SAP_real,DAP_real,sPAP_real,dPAP_real,CO_real
k_Vtot,0.003002,0.007175,0.001197,0.002019,0.003226,0.001059,0.000367
k_Vsys,0.001708,0.000628,0.004516,0.004629,0.000320,0.000680,0.004538
k_Vusv_sys,0.000535,0.011996,0.001852,0.000929,0.001944,0.002690,0.003358
k_Vusv_sys_ven,0.008879,0.001945,0.006721,0.007173,0.012916,0.009015,0.007110
k_Vusv_pulm_ven,0.012773,0.013333,0.016761,0.018237,0.007120,0.010248,0.014808
k_Ctot,0.009182,0.017632,0.006152,0.005961,0.005447,0.006692,0.002235
k_Csys,0.178975,0.390979,0.081449,0.128880,0.168001,0.154390,0.036527
k_Rsysven,0.068238,0.283969,0.027099,0.057957,0.059651,0.078670,0.012183
k_Rpulmart,0.003195,0.256105,0.049896,0.073932,0.081659,0.221512,0.016358
k_ESP_LV,0.743901,0.187697,0.918545,0.928855,0.363912,0.639978,0.874805


In [13]:
compare = pd.DataFrame({
    "phys_mean": phys[tunables].mean(),
    "non_phys_mean": non_phys[tunables].mean(),
    "phys_q01": phys[tunables].quantile(0.01),
    "phys_q99": phys[tunables].quantile(0.99),
    "phys_q05": phys[tunables].quantile(0.05),
    "phys_q95": phys[tunables].quantile(0.95),
})

compare["mean_shift"] = compare["phys_mean"] - compare["non_phys_mean"]

display(compare.sort_values("mean_shift", key=abs, ascending=False))

,phys_mean,non_phys_mean,phys_q01,phys_q99,phys_q05,phys_q95,mean_shift
k_Vtot,72.340075,72.513296,60.181333,84.773704,61.395063,83.615921,-0.173222
k_ESP_RV,1.005483,0.891225,0.933437,1.049341,0.950080,1.046667,0.114258
k_ESP_LV,0.959106,0.895083,0.894233,1.040333,0.904693,1.021984,0.064023
k_Csys,0.870295,0.837480,0.763479,0.928993,0.793538,0.925027,0.032815
k_Rpulmart,0.668326,0.648475,0.504799,0.797705,0.520434,0.787949,0.019851
k_Vusv_pulm_ven,0.882508,0.885207,0.801997,0.967670,0.806357,0.959941,-0.002699
k_Vsys,0.842005,0.839833,0.751734,0.928422,0.757768,0.921362,0.002172
k_Ctot,2.151801,2.149850,1.727778,2.570081,1.758182,2.541812,0.001951
k_Rsysven,0.061451,0.059879,0.048214,0.071824,0.049509,0.071133,0.001572
k_Vusv_sys,0.838933,0.840089,0.751265,0.928324,0.758933,0.920402,-0.001156


Physiological Feasibility Analysis and Iterative Bound Tightening
=================================================================

Objective
---------
The goal of this analysis was to identify physiologically plausible regions of the tunable parameter space for the pediatric DCM pre-VAD cardiovascular model. 

A large-scale sampling campaign of 10,000 simulations was performed on the VSC-5 supercomputer using the Simulink cardiovascular model. The objective was not only to generate simulation data, but also to progressively constrain the uncertain model coefficients toward regions producing physiologically meaningful cardiovascular states.

The selected tunable parameters represent uncertain scaling coefficients controlling:
- vascular volumes,
- venous unstressed volumes,
- systemic compliance,
- vascular resistances,
- and ventricular contractility (ESPVR scaling).

The initial bounds were intentionally broad in order to explore a large region of the parameter space and identify unstable or non-physiological regimes.


Methodology
-----------
1. Broad Initial Sampling
-------------------------
A Latin Hypercube Sampling (LHS) strategy was used to generate 10,000 parameter combinations over wide physiological ranges.

Each simulation produced:
- pressure outputs,
- ventricular volumes,
- cardiac output,
- and additional hemodynamic variables.

The simulations were distributed across the VSC-5 cluster using SLURM job arrays.


2. Physiological Validation
---------------------------
Each simulation was then evaluated against a set of physiological constraints.

The constraints included:
- agreement between target patient values and simulated outputs,
- pressure hierarchy consistency,
- stroke volume conservation,
- pulse pressure constraints,
- and ventricular stability conditions.

A simulation was considered physiologically plausible only if ALL constraints were simultaneously satisfied.

Examples of constraints:
- relative error < 15% for major hemodynamic outputs,
- LVSV ≈ RVSV,
- systemic pulse pressure > 30 mmHg,
- pulmonary pulse pressure > 15 mmHg,
- SAP > CVP,
- mean PAP > LAP.

This filtering step reduced the dataset from 10,000 simulations to 768 physiologically plausible simulations (~7.68%).


3. Analysis of Physiological vs Non-Physiological Simulations
-------------------------------------------------------------
The distributions of tunable parameters were then compared between:
- physiological simulations,
- non-physiological simulations.

The purpose was to identify which parameters strongly influenced physiological feasibility.

The analysis showed that several parameters had very weak influence on physiological validity:
- k_Vtot,
- k_Vsys,
- k_Vusv_sys,
- k_Vusv_sys_ven,
- k_Ctot.

In contrast, four parameters emerged as dominant determinants of physiological plausibility:
- k_ESP_LV,
- k_ESP_RV,
- k_Csys,
- k_Rpulmart.

This result is physiologically coherent:
- ventricular elastance strongly controls pulse pressure generation,
- systemic compliance regulates arterial pressure damping,
- pulmonary arterial resistance and RV elastance strongly influence pulmonary hemodynamics.


4. Bound Tightening Strategy
----------------------------
The physiological simulations were interpreted as an approximation of the feasible physiological manifold of the model.

Rather than using:
- minimum/maximum values,
the tightening process relied on:
- quantile-based bounds.

This avoids:
- overfitting to isolated outliers,
- excessive reduction of exploration diversity.

The following quantiles were analyzed:
- q01–q99 (safe tightening),
- q05–q95 (aggressive tightening).

Because only 7.68% of simulations were physiological, aggressive tightening was considered premature. Therefore, a moderate tightening strategy was selected.

Parameters showing strong physiological discrimination were tightened aggressively.
Parameters with weak discrimination were only slightly tightened or left nearly unchanged.


Interpretation of Results
-------------------------
The low physiological acceptance rate does not indicate model failure.

On the contrary, it demonstrates that:
- the cardiovascular system imposes strong nonlinear physiological constraints,
- many mathematically stable solutions remain physiologically unrealistic.

The purpose of this iterative strategy is therefore:
1. explore broad parameter space,
2. identify physiological regions,
3. progressively concentrate sampling around feasible manifolds,
4. increase density of physiological simulations over successive iterations.

This approach effectively transforms the simulation pipeline into a data-driven physiological calibration framework.


Final Proposed Tunable Parameter Ranges
=======================================

Strongly tightened parameters
-----------------------------

k_ESP_LV
---------
Rationale:
Strong influence on:
- systemic pulse pressure,
- SAP,
- DAP,
- ventricular ejection dynamics.

Final proposed range:
[0.90, 1.03]


k_ESP_RV
---------
Rationale:
Strong influence on:
- pulmonary pressure generation,
- RV stroke volume consistency,
- pulmonary pulse pressure.

Final proposed range:
[0.95, 1.05]


k_Csys
-------
Rationale:
Critical determinant of:
- arterial compliance,
- pulse pressure damping,
- systemic hemodynamic stability.

Final proposed range:
[0.79, 0.925]


k_Rpulmart
-----------
Rationale:
Strongly associated with:
- pulmonary artery pressures,
- RV afterload,
- pulmonary pulse pressure.

Final proposed range:
[0.52, 0.79]


Moderately tightened parameters
-------------------------------

k_Rsysven
----------
Rationale:
Moderate influence on venous return and RAP stability.

Final proposed range:
[0.049, 0.071]


k_Vusv_pulm_ven
----------------
Rationale:
Moderate influence on pulmonary venous reservoir dynamics.

Final proposed range:
[0.806, 0.960]


Lightly tightened / mostly preserved parameters
-----------------------------------------------

k_Vtot
--------
Rationale:
Weak discrimination between physiological and non-physiological simulations.

Final proposed range:
[60.2, 84.8]


k_Vsys
--------
Final proposed range:
[0.752, 0.928]


k_Vusv_sys
------------
Final proposed range:
[0.751, 0.928]


k_Vusv_sys_ven
----------------
Final proposed range:
[0.901, 0.989]


k_Ctot
--------
Rationale:
Total compliance coefficient showed limited discriminative power.

Final proposed range:
[1.73, 2.57]


Conclusion
==========
This first large-scale physiological filtering campaign successfully identified:
- dominant physiological control parameters,
- unstable/non-physiological regions,
- and an approximate feasible manifold for the pediatric DCM configuration.

The next iteration will:
- resample the tightened parameter space,
- increase the density of physiological simulations,
- reduce computational waste,
- and progressively converge toward a robust physiological parameter distribution suitable for:
    - surrogate modeling,
    - inverse calibration,
    - and AI-driven parameter inference.

In [14]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[0]  # if running from 05_Data_Generation notebook

output_dir = (
    ROOT
    / "01_Data"
    / "patient_configs"
    / "pediatric_dcm_patient_01"
)

output_dir.mkdir(parents=True, exist_ok=True)

bounds_df = pd.DataFrame([
    ["k_Vtot", 67.5, 60.2, 84.8, "Lightly tightened; weak discrimination between physiological and non-physiological simulations."],
    ["k_Vsys", 0.84, 0.752, 0.928, "Lightly tightened using q01-q99 physiological range."],
    ["k_Vusv_sys", 0.84, 0.751, 0.928, "Lightly tightened using q01-q99 physiological range."],
    ["k_Vusv_sys_ven", 0.95, 0.901, 0.989, "Lightly tightened using q01-q99 physiological range."],
    ["k_Vusv_pulm_ven", 0.90, 0.806, 0.960, "Moderately tightened; secondary influence on pulmonary venous reservoir dynamics."],
    ["k_Ctot", 2.15, 1.73, 2.57, "Lightly tightened; total compliance showed limited discriminative power."],
    ["k_Csys", 0.85, 0.790, 0.925, "Strongly tightened; critical determinant of systemic compliance and pulse pressure."],
    ["k_Rsysven", 0.060, 0.049, 0.071, "Moderately tightened; influences venous return and RAP stability."],
    ["k_Rpulmart", 2/3, 0.520, 0.790, "Strongly tightened; important for pulmonary artery pressure and RV afterload."],
    ["k_ESP_LV", 0.90, 0.900, 1.030, "Strongly tightened; dominant influence on systemic pressure and LV ejection dynamics."],
    ["k_ESP_RV", 0.90, 0.950, 1.050, "Strongly tightened; dominant influence on pulmonary pressures and RV ejection dynamics."],
], columns=[
    "variable",
    "baseline",
    "lower_bound",
    "upper_bound",
    "tightening_reason",
])

output_path = output_dir / "stage_1_tunable_bounds_v3_physiological.csv"

bounds_df.to_csv(output_path, index=False)

print("Saved bounds to:")
print(output_path)

display(bounds_df)

Saved bounds to:
c:\Users\Maxim\Documents\02_ESILV\A4\Stage\03_Models\DIA\Main_project\01_Data\patient_configs\pediatric_dcm_patient_01\stage_1_tunable_bounds_v3_physiological.csv


,variable,baseline,lower_bound,upper_bound,tightening_reason
0,k_Vtot,67.500000,60.200,84.800,Lightly tightened; weak discrimination between...
1,k_Vsys,0.840000,0.752,0.928,Lightly tightened using q01-q99 physiological ...
2,k_Vusv_sys,0.840000,0.751,0.928,Lightly tightened using q01-q99 physiological ...
3,k_Vusv_sys_ven,0.950000,0.901,0.989,Lightly tightened using q01-q99 physiological ...
4,k_Vusv_pulm_ven,0.900000,0.806,0.960,Moderately tightened; secondary influence on p...
5,k_Ctot,2.150000,1.730,2.570,Lightly tightened; total compliance showed lim...
6,k_Csys,0.850000,0.790,0.925,Strongly tightened; critical determinant of sy...
7,k_Rsysven,0.060000,0.049,0.071,Moderately tightened; influences venous return...
8,k_Rpulmart,0.666667,0.520,0.790,Strongly tightened; important for pulmonary ar...
9,k_ESP_LV,0.900000,0.900,1.030,Strongly tightened; dominant influence on syst...
